# PhysicalVarNet: Physics-Driven Scientific Workflow
This notebook implements the training and validation pipeline for PhysicalVarNet. It is organized into a clean, modular structure: Imports -> Utilities -> Data -> Setup -> Train -> Visualize -> Validate.

## 1. Imports and Environment Setup

In [ ]:
import os
import sys
import yaml
import torch
import pathlib
import h5py
import json
import numpy as np
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
from pathlib import Path
from types import SimpleNamespace
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Ensure custom fastmri library is prioritized in path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.resolve()
src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from fastmri_custom.data import mri_data, subsample, transforms
from fastmri_custom.data.transforms import VarNetDataTransform
from fastmri_custom.data.mri_data import CombinedSliceDataset, SliceDataset
from fastmri_custom.pl_modules.varnet_module import VarNetModule
import fastmri_custom as fastmri
from fastmri_custom import fftc

def load_config(path='../configs/config.yaml'):
    """Loads the project configuration."""
    if not os.path.exists(path):
        if os.path.exists('configs/config.yaml'):
            path = 'configs/config.yaml'
    with open(path, 'r') as f:
        return yaml.safe_load(f)

config = load_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

## 2. Important Functions
Core utilities for GPU memory management and device alignment.

In [ ]:
def load_batches_to_device(iterator, num_batches, device):
    """
    Loads specified number of batches directly to GPU into a SimpleNamespace.
    """
    batch_list = []
    for _ in range(num_batches):
        try:
            batch = next(iterator)
            b_dev = SimpleNamespace(
                masked_kspace=batch.masked_kspace.to(device),
                mask=batch.mask.to(device),
                num_low_frequencies=batch.num_low_frequencies,
                target=batch.target.to(device),
                tr=batch.tr.to(device) if torch.is_tensor(batch.tr) else torch.tensor(batch.tr).to(device),
                te=batch.te.to(device) if torch.is_tensor(batch.te) else torch.tensor(batch.te).to(device),
                alpha=batch.alpha.to(device) if torch.is_tensor(batch.alpha) else torch.tensor(batch.alpha).to(device),
                contrast=batch.contrast.to(device).float() if torch.is_tensor(batch.contrast) else torch.tensor(batch.contrast).to(device).float(),
                t1_gt=batch.t1_gt.to(device),
                t2_gt=batch.t2_gt.to(device),
                pd_gt=batch.pd_gt.to(device),
                max_value=batch.max_value.to(device) if torch.is_tensor(batch.max_value) else batch.max_value,
                fname=batch.fname
            )
            batch_list.append(b_dev)
        except StopIteration: break
    return batch_list

def release_batches_from_gpu(batch_list):
    """Clears GPU tensors and empties cache."""
    for b in batch_list:
        for attr in list(vars(b).keys()): delattr(b, attr)
    batch_list.clear()
    torch.cuda.empty_cache()

def check_device_alignment(model, batch):
    print(f"{'='*20} DEVICE ALIGNMENT {'='*20}")
    model_params = list(model.parameters())
    all_on_cuda = all(p.is_cuda for p in model_params)
    print(f"Model State: {'✅ CUDA' if all_on_cuda else '❌ MIXED/CPU'}")
    for attr in vars(batch):
        item = getattr(batch, attr)
        if torch.is_tensor(item):
            print(f"  {attr:20}: {'✅ CUDA' if item.is_cuda else '❌ CPU'}")
    print(f"{'='*54}")

## 3. Upload Datasets
Initialization of physics-aware data loaders.

In [ ]:
train_path = Path(config['paths'].get('train_path', '../data/train'))
val_path = Path(config['paths'].get('val_path', '../data/val'))

mask_func = subsample.RandomMaskFunc(center_fractions=[0.08], accelerations=[4])
physics_transform = VarNetDataTransform(mask_func=mask_func)

dataset_train = SliceDataset(root=train_path, challenge='multicoil', transform=physics_transform)
dataset_val = SliceDataset(root=val_path, challenge='multicoil', transform=physics_transform)

train_loader = DataLoader(dataset_train, batch_size=1, shuffle=True)
val_loader = DataLoader(dataset_val, batch_size=1, shuffle=False)

print(f"Dataset Ready: {len(dataset_train)} training and {len(dataset_val)} validation slices.")

## 4. Prepare for Training
Model initialization and optimizer setup.

In [ ]:
model = VarNetModule(num_cascades=12, chans=18, pools=4)
model.to(device)

opt_config = model.configure_optimizers()
optimizer = opt_config['optimizer']
scheduler = opt_config['lr_scheduler']['scheduler']

save_dir = Path("../checkpoints")
save_dir.mkdir(exist_ok=True)
print("Model and Optimizer initialized.")

## 5. Training Loop
Iterative training with interleaved validation steps.

In [ ]:
num_epochs = 15
train_buffer = 16
val_buffer = 4

for epoch in range(num_epochs):
    model.train()
    train_iterator = iter(train_loader)
    val_iterator = iter(val_loader)
    num_steps = (len(train_loader) + train_buffer - 1) // train_buffer
    
    for step in range(num_steps):
        t_batches = load_batches_to_device(train_iterator, train_buffer, device)
        if not t_batches: break
        
        for batch in t_batches:
            optimizer.zero_grad()
            loss = model.training_step(batch, epoch)
            if torch.isfinite(loss):
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
        
        # Validation interleave
        model.eval()
        v_batches = load_batches_to_device(val_iterator, val_buffer, device)
        with torch.no_grad():
            for vb in v_batches: _ = model.training_step(vb, epoch)
        
        release_batches_from_gpu(t_batches)
        release_batches_from_gpu(v_batches)
        
        if step % 10 == 0: print(f"Epoch {epoch+1} Step {step}/{num_steps} | Loss: {loss.item():.6f}")
    
    torch.save(model.state_dict(), save_dir / "last_model.pth")
    print(f">>>> End of Epoch {epoch+1} <<<<")

## 6. Visualization
Comprehensive results inspection: Reconstruction vs GT and T1/T2/PD Mapping.

In [ ]:
def show_results(batch, model):
    model.eval()
    with torch.no_grad():
        output, tissue_maps, params = model.varnet(batch.masked_kspace, batch.mask, batch.num_low_frequencies)
    
    # Compute zero-filled reconstruction for comparison
    zf = fftc.ifft2c_new(batch.masked_kspace)
    zf_abs = fastmri.complex_abs(zf).sum(dim=1)
    _, zf_cropped = transforms.center_crop_to_smallest(batch.target, zf_abs)
    
    gt = batch.target[0].cpu().numpy()
    rec = output[0].cpu().numpy()
    corrupted = zf_cropped[0].cpu().numpy()
    
    # Plot 1: Reconstructions
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 4, 1); plt.imshow(corrupted, cmap='gray'); plt.title("Zero-Filled (Corrupted)"); plt.axis('off')
    plt.subplot(1, 4, 2); plt.imshow(rec, cmap='gray'); plt.title("PhysicalVarNet"); plt.axis('off')
    plt.subplot(1, 4, 3); plt.imshow(gt, cmap='gray'); plt.title("Ground Truth"); plt.axis('off')
    plt.subplot(1, 4, 4); plt.imshow(np.abs(gt - rec), cmap='hot'); plt.title("Error Map"); plt.axis('off')
    plt.show()
    
    # Plot 2: Tissue Maps (T1, T2, PD)
    if tissue_maps is not None:
        t1, t2, pd = tissue_maps[0,0].cpu().numpy(), tissue_maps[0,1].cpu().numpy(), tissue_maps[0,2].cpu().numpy()
        plt.figure(figsize=(18, 5))
        plt.subplot(1, 3, 1); plt.imshow(t1, cmap='viridis'); plt.title("Predicted T1"); plt.colorbar(shrink=0.8); plt.axis('off')
        plt.subplot(1, 3, 2); plt.imshow(t2, cmap='plasma'); plt.title("Predicted T2"); plt.colorbar(shrink=0.8); plt.axis('off')
        plt.subplot(1, 3, 3); plt.imshow(pd, cmap='inferno'); plt.title("Predicted PD"); plt.colorbar(shrink=0.8); plt.axis('off')
        plt.show()

sample = load_batches_to_device(iter(val_loader), 1, device)[0]
show_results(sample, model)

## 7. Final Validation and MSE Report
Quantitative assessment across the full validation set.

In [ ]:
model.eval()
mse_history = []
v_iterator = iter(val_loader)
with torch.no_grad():
    for _ in range(len(dataset_val)):
        batches = load_batches_to_device(v_iterator, 1, device)
        if not batches: break
        out, _, _ = model.varnet(batches[0].masked_kspace, batches[0].mask, batches[0].num_low_frequencies)
        mse_history.append(F.mse_loss(out, batches[0].target).item())
        release_batches_from_gpu(batches)

print("========================================")
print(f"Final Image Mean MSE: {np.mean(mse_history):.8f}")
print("========================================")